In [1]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.sparse.linalg import eigsh
from sympy import Matrix, nsimplify
from IPython.display import display

from sqoct_core import (
    onsite_clean,
    onsite_impurity,
    hopping_bdg,
    build_ysr_system
)

In [2]:
from IPython.display import display, HTML
from IPython.core.magic import register_cell_magic
import uuid

display(HTML("""
<style>
.tutorial-note {
    padding: 12px 16px;
    margin: 12px 0;
    border-left: 5px solid #4c8bf5;
    background: rgba(76, 139, 245, 0.08);
    border-radius: 4px;
}

.tutorial-important {
    padding: 12px 16px;
    margin: 12px 0;
    border-left: 5px solid #e67e22;
    background: rgba(230, 126, 34, 0.08);
    border-radius: 4px;
}

.tutorial-warning {
    padding: 12px 16px;
    margin: 12px 0;
    border-left: 5px solid #d9534f;
    background: rgba(217, 83, 79, 0.08);
    border-radius: 4px;
}

.tutorial-result {
    padding: 12px 16px;
    margin: 12px 0;
    border-left: 5px solid #2ca02c;
    background: rgba(44, 160, 44, 0.08);
    border-radius: 4px;
}
</style>
"""))



@register_cell_magic
def important(line, cell):
    marker = f"important-{uuid.uuid4().hex}"

    display(HTML(f"""
    <div id="{marker}"></div>

    <style>
    .jp-CodeCell:has(#{marker}) {{
        background: rgba(230, 126, 34, 0.08);
        border-left: 5px solid #e67e22;
        border-radius: 5px;
        padding-left: 6px;
    }}
    </style>
    """))

    get_ipython().run_cell(cell)

<div class="tutorial-note">
    Once we can construct the Hamiltonian we can look at Low-energy YSR spectrum.
    So we now build the square-octagon Hamiltonian directly and look at its YSR states.
</div>

In [3]:
Nx = 2
Ny = 2

t1 = 1.0
t2 = 0.5

mu = 0.3
Delta = 0.2
J = 0.4


fsyst, impurity_site, impurity_index = build_ysr_system(
    Nx, Ny,
    t1=t1, t2=t2,
    impurity_sublattice="A"
)

print("Number of sites =", len(fsyst.sites))
print("Hamiltonian dimension =", 4 * len(fsyst.sites))
print("Impurity site =", impurity_site)
print("Impurity index =", impurity_index)

Number of sites = 16
Hamiltonian dimension = 64
Impurity site = <Site [1 1] of sqo0>
Impurity index = 11


In [4]:
H = fsyst.hamiltonian_submatrix(
    params={
        "J": J,
        "mu": mu,
        "Delta": Delta
    },
    sparse=False
)

evals_full = np.linalg.eigvalsh(H)

print(evals_full)

[-3.40867181 -3.3730304  -3.00149477 -2.95303197 -2.95303197 -2.91418232
 -2.42286363 -2.35743874 -1.92763784 -1.9118359  -1.55730148 -1.52104876
 -1.52104876 -1.50029891 -1.00165903 -0.90228294 -0.82630514 -0.79332527
 -0.67895735 -0.58872478 -0.58872478 -0.55839618 -0.55823123 -0.54570654
 -0.54075268 -0.44899421 -0.43591883 -0.37338451 -0.37338451 -0.34156224
 -0.31416079 -0.23713858  0.23713858  0.31416079  0.34156224  0.37338451
  0.37338451  0.43591883  0.44899421  0.54075268  0.54570654  0.55823123
  0.55839618  0.58872478  0.58872478  0.67895735  0.79332527  0.82630514
  0.90228294  1.00165903  1.50029891  1.52104876  1.52104876  1.55730148
  1.9118359   1.92763784  2.35743874  2.42286363  2.91418232  2.95303197
  2.95303197  3.00149477  3.3730304   3.40867181]


    We look only at eigenvalues closest to zero

In [5]:
order = np.argsort(np.abs(evals_full))

evals_near_zero = evals_full[order[:8]]

print(evals_near_zero)

[ 0.23713858 -0.23713858 -0.31416079  0.31416079  0.34156224 -0.34156224
 -0.37338451  0.37338451]


    For practical purposes we construct the same Hamiltonian in sparse form. 

In [6]:
H_sparse = fsyst.hamiltonian_submatrix(
    params={
        "J": J,
        "mu": mu,
        "Delta": Delta
    },
    sparse=True
)

# sigma=1e-7 tells eigsh to search near zero energy.
# shift-invert, which="LM" then means the eigenvalues effectively closest to sigma, not the largest-energy eigenvalues of the original Hamiltonian.
evals_sparse = eigsh(
    H_sparse,
    k=6,
    sigma=1e-7,
    which="LM",
    return_eigenvectors=False
)

print(evals_sparse)

[-0.34156224  0.34156224 -0.31416079  0.31416079 -0.23713858  0.23713858]


In [7]:
evals_sparse = np.sort(evals_sparse)

print(evals_sparse)

[-0.34156224 -0.31416079 -0.23713858  0.23713858  0.31416079  0.34156224]


    So we can simply turn the above calculations into a function

In [8]:
%%important

def low_energy_states(
    fsyst, J, mu, Delta,
    k=6, sigma=1e-7
):

    H = fsyst.hamiltonian_submatrix(
        sparse=True,
        params={
            "J": J,
            "mu": mu,
            "Delta": Delta
        }
    )

    vals = eigsh(H, k=k, sigma=sigma, which="LM", return_eigenvectors=False)

    return np.sort(vals)

In [ ]:
vals = low_energy_states(fsyst, J=J, mu=mu, Delta=Delta)

print(vals)

<div class="tutorial-note">

The YSR energy is simply

$$
E_{\rm YSR}
=
\min_n |E_n|.
$$

</div>

In [9]:
%%important
def ysr_energy(fsyst, J, mu, Delta, k=6):

    vals = low_energy_states(fsyst, J=J, mu=mu, Delta=Delta, k=k)

    return np.min(np.abs(vals))

In [ ]:
E = ysr_energy(
    fsyst,
    J=0.4,
    mu=mu,
    Delta=Delta
)

print("E_YSR =", E)

<div class="tutorial-note">

distinction between what `low_energy_states()` returns and what `ysr_energy()` does. `ysr_energy()` deliberately discards the sign and reports the distance of the closest BdG state from zero.

</div>